# P1: 회귀 — 실습 노트북

42H 머신러닝 강의노트의 **P1(2~4주, 회귀 프로젝트)** 실습용 코드 모음이다.
캘리포니아 주택가격 데이터셋을 이용한 머신러닝 프로젝트 전체 과정(실습1·실습2)과
선형회귀 모델의 학습 원리(실습3)를 하나로 이어서 다룬다.

**감사의 글**

오렐리앙 제롱<font size='2'>Aurélien Géron</font>의 [Hands-On Machine Learning with Scikit-Learn and PyTorch (O'Reilly, 2025)](https://github.com/ageron/handson-mlp)에 사용된 코드를 참고한 실습 노트북이다. 보다 심화된 이해를 위해 책 원본을 읽을 것을 강력하게 권장한다. 자료를 공개한 오렐리앙 제롱과 일부 그림 자료를 제공해 준 한빛아카데미에게 진심어린 감사를 전한다.

**권장 사항**

[(강의노트) 머신러닝 프로젝트](https://codingalzi.github.io/code-workout-ml/end2end-ml-project/)와
[(강의노트) 모델 훈련](https://codingalzi.github.io/code-workout-ml/training-models/)을 병행하여 읽을 것을 권장한다.

## 환경설정

This project requires Python 3.10 or above:

In [ ]:
import sys

assert sys.version_info >= (3, 10)

It also requires Scikit-Learn ≥ 1.6.1:

In [ ]:
from packaging.version import Version
import sklearn

assert Version(sklearn.__version__) >= Version("1.6.1")

Let's define the default font sizes, to plot pretty figures:

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=12)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=12)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

**기본 라이브러리**

In [ ]:
import numpy as np
import pandas as pd

## 머신러닝 모델 훈련용 데이터

### 데이터 기초 정보

1990년도에 시행된 미국 캘리포니아 주의 20,640개 구역별 주택 가격 데이터는
경도, 위도, 주택 건물 중위연령, 총 방 수, 총 침실 수, 인구, 가구 수, 중위소득, 중위 주택가격, 해안 근접도
등 총 10개의 **특성**<font size='2'>feature</font>을 포함한다.
참고로 통계 분야에서는 특성을 변수 또는 변인 등으로 부르지만 머신러닝 분야에서는 특성이라 부르는 게 일반적이다.

아래 그림은 원본 csv 파일의 일부 내용을 보여준다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/housing-data.png?raw=true" width="700">
</div>

### 머신러닝 훈련 모델 선택 기준

구역별 중위 주택가격을 타깃으로 예측하는 시스템에 활용될
**회귀 모델**을 훈련시키고자 한다.
여기서 훈련시킬 모델의 특성은 다음과 같다.

* 지도학습: 구역별 중위 주택가격을 타깃, 즉
    최대한 정확하게 예측해야 하는 목표로 지정한다.

* 회귀: 중위 주택가격, 즉 이산형 값이 아닌 연속형 값을 예측한다.
    보다 세분화하면 다중 회귀이자 단변량 회귀 모델이다.
  * 다중 회귀<font size="2">multiple regression</font>: 구역별로 여러 특성을 주택 가격 예측에 사용
  * 단변량 회귀<font size="2">univariate regression</font>: 구역별로 한 종류의 값만 예측

* 배치 학습: 빠르게 변하는 데이터에 적응할 필요가 없으며, 데이터셋의 크기도 충분히 작기에
    데이터셋 전체를 대상으로 훈련을 진행한다.

### 데이터 구하기

캘리포니아 주택 가격 데이터는 매우 유명하여 많은 공개 저장소에서 다운로드할 수 있다.
여기서는 개인 깃허브 리포지토리에 압축파일로 저장한 파일을 다운로드해서 사용한다.

아래 코드의 `load_housing_data()` 함수는
지정된 깃허브 리포지토리에 tgz 형식의 압축 파일로 저장되어 있는
캘리포니아 주택 가격 데이터를 다운로드한 후에
Pandas 데이터프레임으로 변환하여 반환한다.
최종적으로 `housing_full` 변수는 캘리포니아 주택 가격 데이터를 담고 있는 데이터프레임을 가리킨다.

**데이터셋 다운로드**

In [ ]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets", filter="data")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing_full = load_housing_data()

## 데이터셋 탐색

데이터 탐색은 훈련을 시작하기 전에 주어진 데이터셋의
다양한 특성을 대략적으로 살펴보는 과정이다.
데이터 시각화는 데이터 탐색을 위한 주요 기법 중의 하나이다.
먼저 데이터프레임의 메서드를 활용한 데이터 탐색을 알아본다.

### 데이터프레임 활용

In [ ]:
housing_full.head()

In [ ]:
housing_full.info()

### 범주형 대 수치형

**범주형 특성**

'해안 근접도'는 5개의 범주로 구분된다.
데이터프레임의 `valule_counts()` 메서드는 사용된 특성값과 각각의 특성값이 사용된 횟수를 확인해준다.

In [ ]:
housing_full["ocean_proximity"].value_counts()

각 특성값의 의미는 다음과 같다.

| 특성값 | 설명 |
| :--- | :--- |
| <1H OCEAN | 해안에서 1시간 이내 |
| INLAND | 내륙 |
| NEAR OCEAN | 해안 근처 |
| NEAR BAY | 샌프란시스코의 Bay Area 구역 |
| ISLAND | 섬  |

**수치형 특성**

범주형 특성과는 다르게 수치형 특성은 정수 또는 부동소수점으로 구성된 특성이며,
평균값, 표준편차, 사분범위 등 수치형 특성들의 정보를 확인할 수 있다.
해안 근접도를 제외한 나머지 특성들 모두 수치형 특성이다.

In [ ]:
housing_full.describe()

수치형 특성별로 히스토그램을 통해 다음 정보를 얻을 수 있다.

- 각 특성마다 사용되는 단위와 스케일이 다르다. 1 단위부터 만 단위까지 다양하다.
- 일부 특성은 한쪽으로 치우쳐저 있다.
    예를 들어 `total_rooms`, `total_bedrooms`, `population`, `households` 등의 특성값들이 오른쪽 꼬리를 길게 갖는다.
- 일부 특성은 값을 제한한 것으로 보인다.
 예를 들어 `housing_median_age`, `median_house_value` 등의 특성값 상한값이 임의로 지정되어 잘린 것처럼 보인다.

In [ ]:
housing_full.hist(bins=50, figsize=(12, 8))

plt.show()

## 훈련셋과 테스트셋

모델 훈련을 시작하기 전에 전체 데이터셋을 보통 **훈련셋**<font size='2'>training set</font>과
**테스트셋**<font size='2'>test set</font>으로 나눈다.

테스트셋은 훈련 과정에서 전혀 사용하지 않는 데이터이며, 보통 전체 데이터셋의 약 10~20% 정도를 차지하도록 정한다.
다만 전체 데이터셋의 크기에 따라 테스트셋의 비율은 적절히 조절할 수 있다.

- **훈련셋**: 머신러닝 모델을 훈련하는 데 사용하는 데이터셋이다.
  실제 모델 훈련을 시작하기 전에 입력 데이터셋과 타깃 데이터셋으로 다시 나눈다.

- **테스트셋**: 훈련을 마친 모델의 성능을 평가하기 위해 사용하는 데이터셋이다.
  훈련 과정에서는 어떤 방식으로도 사용하지 않는다.
  모델 평가를 진행하기 전에 훈련셋과 동일한 기준에 따라 입력 데이터셋과 타깃 데이터셋으로 나눈다.

### 무작위 샘플링

전체 데이터에서 무작위로 샘플을 추출하는 방식이다.
데이터셋이 매우 크다면 모집단을 잘 대표할 수 있지만, 그렇지 않을 경우 샘플링 편향이 발생해 특정 특징을 가진 데이터가 과하게 많거나 적게 추출될 위험이 있다.

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(housing_full, test_size=0.2,
                                       random_state=42)

### 계층 샘플링

샘플링 편향을 최대한 방지하기 위해 중위소득을 기준으로
계층 샘플링<font size='2'>stratified sampling</font>을 활용할 수 있다.
계층 샘플링을 층화표집으로 부르기도 한다.

먼저 구역별 중위소득 특성을 대상으로 히스토그램을 그려보면
대부분 구역의 중위소득이 1.5 ~ 6.0, 즉 15,000에서 60,000 달러 사이인 것을 알 수 있다.

In [ ]:
housing_full['median_income'].hist()

따라서 중위소득 구간을 아래처럼 5개로 구분한 다음에 계층 샘플링을 이용하여
훈련셋과 테스트셋을 구분하면 좋을 것 같아 보인다.

In [ ]:
housing_full["income_cat"] = pd.cut(housing_full["median_income"],
                                    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                                    labels=[1, 2, 3, 4, 5])

먼저 언급된 5 개의 구간으로 구분하는 `"income_cat"` 특성을 추가한다.
`pd.cut()` 함수는 특성과 구간 구분이 주어지면 각 구간에 해당하는 특성값들에 지정된 레이블을 할당하는
형식으로 새로운 특성값들을 만들어 낸다.

In [ ]:
cat_counts = housing_full["income_cat"].value_counts().sort_index()
cat_counts.plot.bar(rot=0, grid=True)
plt.xlabel("Income category")
plt.ylabel("Number of districts")

plt.show()

이제 `"income_cat"` 특성에 사용된 값들의 분포 비율을 반영하면서
훈련셋과 테스트셋을 8대 2로 나눈다.
사이킷런의 `train_test_split()` 함수는 데이터프레임에 속한 샘플을
지정된 비율로 두 개의 데이터프레임으로 나눌 때
계층 샘플링을 지원한다.

In [ ]:
strat_train_set, strat_test_set = train_test_split(
    housing_full,
    test_size=0.2,
    stratify=housing_full["income_cat"],            # 계층 샘플링을 위한 기준이 되는 열 지정
    random_state=42)

### 무작위 샘플링 대 계층 샘플링

계층 샘플링이 무작위 샘플링보다 계층별 샘플의 비율을 훨씬 잘 유지함을 아래 표가 확인해준다.
`Strat. Error %`와 `Rand. Error %`는 모집단에서의 계층별 빈도 기준 표본에서의 계층별 빈도의
변화율을 가리킨다.
계층 샘플링이 변화율이 무작위 샘플링의 변화율보다 작음에 주목한다.

In [ ]:
# extra code – computes the data for Figure 2–10

def income_cat_proportions(data):
    return data["income_cat"].value_counts() / len(data)

train_set, test_set = train_test_split(housing_full, test_size=0.2,
                                       random_state=42)

compare_props = pd.DataFrame({
    "Overall %": income_cat_proportions(housing_full),
    "Stratified %": income_cat_proportions(strat_test_set),
    "Random %": income_cat_proportions(test_set),
}).sort_index()

compare_props.index.name = "Income Category"
compare_props["Strat. Error %"] = (compare_props["Stratified %"] /
                                   compare_props["Overall %"] - 1)
compare_props["Rand. Error %"] = (compare_props["Random %"] /
                                  compare_props["Overall %"] - 1)
(compare_props * 100).round(2)

**`income_cat` 특성 삭제**

해당 특성은 계층 샘플링을 위한 용도로만 사용되기에 더 이상 필요 없어서
훈련셋과 테스트셋 모두에서 삭제한다.

In [ ]:
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

## 훈련셋 살펴보기

데이터 살펴보기는 훈련셋만을 대상으로 진행한다.
이유는 머신러닝 모델에 사용될 좋은 훈련셋으로 활용하는 방안을 모색하기 위해서인데,
테스트셋을 사용하면 미래에 발생할 데이터를 안다고 가정하는 결과를 초래하기 때문이다.

훈련셋은 캘리포니아 전체 데이터셋에서 앞서 계층 샘플링을 이용하여 얻어진 데이터셋으로 지정한다.
아래 코드는 `housing` 변수가 가리키는 데이터셋 객체를 전체 데이터셋에서
계층 샘플링을 이용하여 생성된 훈련셋의 복사본으로 지정한다.

In [ ]:
housing = strat_train_set.copy()

훈련셋의 크기는 전체 데이터셋의 크기인 20,640의 80%인 16,512다.

In [ ]:
housing.shape

### 훈련셋 시각화

훈련셋에 포함된 16,512개 구역의
경도와 위도 정보를 이용하여 구역 정보를 산포도로 나타내면 인구의 밀집 정도를 확인할 수 있다.
예를 들어, 샌프란시스코의 Bay Area, LA, 샌디에고 등 유명 대도시의 특정 구역이 높은 인구 밀도를 갖는다.

데이터프레임의 `plot()` 메서드는 다양한 종류의 그래프를 그린다.
`kind` 매개변수의 키워드 인자를 `"scatter"`로 지정하면 산점도를 그린다.
이외에 다양한 키워드 인자를 그래프 옵션으로 지정할 수 있다.
예를 들어 구역의 중위 주택가격을 색상으로,
인구밀도는 원의 크기로 활용하면
인구 밀도가 높은 유명 대도시의 특정 구역에 위치한
주택 가격이 높다는 일반적인 사실 또한 쉽게 확인된다.

In [ ]:
housing.plot(kind="scatter",
             x="longitude",
             y="latitude",
             grid=True,
             s=housing["population"] / 100, label="population",
             alpha=0.6,
             legend=True,
             sharex=False,
             figsize=(10, 7))

plt.show()

아래 그래프는 구역의 중위 주택가격을 색상으로 함께 표시한 것이다.
인구 밀도가 높은 대도시 구역일수록 주택가격이 높은 경향이 뚜렷하게 확인된다.
이 성질은 뒤에서 위도·경도 특성을 전처리할 때 참고한다.

In [ ]:
housing.plot(kind="scatter",
             x="longitude",
             y="latitude",
             grid=True,
             s=housing["population"] / 100, label="population",
             c="median_house_value",
             cmap="jet", colorbar=True,
             alpha=0.6,
             legend=True,
             sharex=False,
             figsize=(10, 7))

plt.show()

### 피어슨 상관관계

앞으로 훈련시킬 모델은 어떤 구역의 중위 주택가격을 제외한 다른 특성이 주어졌을 때
해당 구역의 중위 주택가격을 예측해야 한다.
따라서 중위 주택가격과 상관관계가 높은 특성을 미리 확인해볼 필요가 있다.

특성들 사이의 선형 상관관계를 피어슨 상관계수로 계산한다.
단, 수치형 특성만 대상으로 한다.

In [ ]:
corr_matrix = housing.corr(numeric_only=True)

Seaborn 라이브러리의 히트맵 함수를 이용하여 상관관계의 강도를 시각화해본다.

In [ ]:
import seaborn as sns
sns.heatmap(corr_matrix,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            annot_kws={"size": 9})
plt.show()

중위 주택가격과 중위소득의 상관계수가 0.68로 상당히 높다.
이는 중위소득이 올라가면 중위 주택가격도 상승하는 경향이 꽤 강하게 있음을 의미한다.
아래 산점도가 이 사실을 잘 확인시켜준다.

In [ ]:
housing.plot(kind="scatter", x="median_income", y="median_house_value",
             alpha=0.1, grid=True)

plt.show()

## 타깃 대 입력 데이터셋

지도학습 방식으로 중위 주택가격을 예측하는 모델을 훈련시키려면
훈련셋을 타깃셋과 입력 데이터셋으로 분리해야 한다.
입력 데이터셋 또한 일반적으로 훈련셋으로 불린다.

In [ ]:
# 훈련셋 (입력 데이터셋)
housing = strat_train_set.drop("median_house_value", axis=1)

# 타깃셋
housing_labels = strat_train_set["median_house_value"].copy()

**참고:**

- 이전 셀의 코드는 `housing` 데이터프레임 새롭게 정의한다.
    이렇게 하는 이유는 입력 데이터셋과 타깃셋을 구분하는 과정을 포함하여 앞으로 설명할 모든 전처리 과정을 한 번에 처리할 자동화된 '변환 파이프라인'을 구축하기 위함이다.

- `strat_train_set.drop()` 함수 또한 새로운 객체를 생성함에 주목한다.
    단, `inplace=True` 키워드 인자를 사용하지 않아야 한다.

## 데이터 정제와 전처리

데이터 탐색을 통해 확인한 지도학습 회귀 모델을 훈련시키기 위해 먼저
적절한 훈련셋(입력 데이터셋)을 준비해야 한다.
적절한 훈련셋(입력 데이터셋) 준비는 데이터 정제와 데이터 전처리 과정으로 이루어진다.

### 데이터 정제

먼저 데이터 정제<font size='2'>Data Cleanign</font>는
일반적으로 훈련셋(입력 데이터셋)에 포함된 결측치 처리, 이상치와 노이즈 제거 등을 의미한다.
캘리포니아 주택 가격 데이터셋의 경우 구역별 총 방 수를 의미하는 `total_rooms` 특성에
포함되어 있는 결측치를 어떻게 다를 것인지 결정해야 한다.
이상치와 노이즈에 대해서는 여기서는 다루지 않는다.

### 데이터 전처리

데이터 전처리<font size='2'>Data Preprocessing</font>는
모델 훈련에 적합한 훈련셋(입력 데이터셋)을 만들어가는 과정을 가리킨다.
예를 들어, 캘리포니아 주택 가격 데이터셋에 포함된 수치형 특성과 범주형 특성에 대해
각각 아래 전처리 과정을 거친다.

* 범주형 특성 전처리: 원-핫-인코딩
* 수치형 특성 전처리: 특성 스케일링과 특성 조합

이에 더해 다음 전처리 과정도 진행한다.

- 비율 특성 추가: 침실 비율, 가구당 방 수, 가구당 평균 가구원 수
- 로그 변환 대상 특성: `"total_bedrooms"`, `"total_rooms"`, `"population"`, `"households"`, `"median_income"`

## 사이킷런 API 활용

데이터 정제와 전처리 전과정을 사이킷런 라이브러리에서 제공하는 API를 활용한다.
먼저 사이킷런 API의 기본 특성을 살펴본 다음에 앞서 언급된 정제와 전처리 내용을
처리하는 각각의 API를 하나씩 살펴본다.
그런 다음 사이킷런 API를 연동하여 정제와 전처리 전 과정을
한꺼번에 순차적으로 처리하는 파이프라인<font size='2'>pipeline</font>으로 구성하여
자동화는 방식까지 소개한다.

**주의 사항**

여기서는 사이킷런 API를 활용한 데이터 정제와 전처리 사용법을 구체적인 예제를 이용하여 소개한다.
하지만 실제 캘리포니아 주택 가격 데이터셋의 변환은
맨 나중에 여기서 소개된 API들 조합하여
한꺼번에 처리할 예정이다.

### 결측치 처리

`'total_bedrooms'` 특성에 결측치(`NaN`)가 일부 포함되어 있다.

**결측치가 하나라도 포함된 행 확인**

- `total_bedrooms`에만 총 168개의 결측치 포함됨.
- `null_rows_idx`: 결측치가 위치한 행만 선택하기 위한 부울 마스크

In [ ]:
null_rows_idx = housing.isnull().any(axis=1)
housing.loc[null_rows_idx]

아래 코드로도 확인 가능

In [ ]:
housing.loc[null_rows_idx].info()

**SimpleImputer 변환기**

`SimpleImputer` 변환기를 활용하여 수치형 결측치를 해당 특성의 중앙값<font size='2'>median</font>으로 대체한다.
아래 코드는 결측치를 해당 특성의 중앙값으로 대체하는 기능을 갖는
`SimpleImputer` 변환기 객체를 `imputer`에 할당한다.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

`SimpleImputer` 변환기는 수치형 특성에 대해서만 작동한다.
아래 코드는 해안 근접도 특성을 제외한 나머지 수치형 특성만으로
구성된 데이터프레임을 대상으로 모든 결측치를 해당 특성의 중앙값으로
대체하기 위해 필요한 특성별 중앙값을 찾아 `imputer`가 가리키는 변환기 객체에 저장한다.

In [ ]:
housing_num = housing.select_dtypes(include=[np.number])
imputer.fit(housing_num)

`fit()` 메서드가 찾아낸 특성별 중앙값 정보는 변환기 객체의 `statistics_` 속성에 저장된다.

In [ ]:
imputer.statistics_

`imputer` 변환기 객체의 `transform()` 메서드는 앞서 찾아낸 특성별 중앙값를 이용하여 각 특성에 포함된 모든 결측치를 대체한 새로운 **넘파이 어레이**를 생성한다.

In [ ]:
X = imputer.transform(housing_num)

In [ ]:
type(X)

`fit_transform()` 메서드는 `fit()` 메서드와 `transform()` 메서드를 연속으로 호출하기에
`X`를 다음과 같이 바로 생성할 수 있다.

In [ ]:
X = imputer.fit_transform(housing_num)

데이터프레임이 아닌 넘파이 어레이를 생성하는 과정에서 잊혀진 특성 이름은
변환기 객체의 `feature_names_in_` 속성으로 저장되어 있다.

In [ ]:
imputer.feature_names_in_

특성명을 활용하여 다시 데이터프레임으로 변환한다.

In [ ]:
housing_tr = pd.DataFrame(X,
                          columns=housing_num.columns,
                          index=housing_num.index)

`total_bedrooms`의 모든 결측치가 해당 특성의 중앙값로 대체되었음을 확인할 수 있다.

In [ ]:
housing_tr.loc[null_rows_idx]

결측치를 대체하는 전략은 변환기 객체의 `strategy` 속성에 저장되어 있다.

In [ ]:
imputer.strategy

참고로 `SimpleImputer`의 객체를 생성할 때 사용할 수 있는 `strategy`는 다음과 같다.

- `"mean"`: 평균값으로 채움 (수치형 데이터 전용) - 기본값
- `"median"`: 중간값(중앙값)으로 채움 (수치형 데이터 전용)
- `"most_frequent"`: 최빈값(가장 자주 등장하는 값)으로 채움 (수치형/문자열 데이터 모두 가능)
- `"constant"`: 지정한 상수로 채움.
    `strategy` 매갭변수에 대한 키워드 인자 이외에 `fill_value` 매개변수의 키워드 인자로 지정

    ```python
    SimpleImputer(strategy="constant", fill_value="채울값")
    ```

### 원-핫 인코딩

해안 근접도(`ocean_proximity`)는 문자열을 사용한다.

In [ ]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.head(8)

사이킷런의 `OneHotEncoder` 변환기가 원-핫 인코딩을 지원한다.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)

결과는 **희소 행렬**<font size='2'>sparse array</font> 형식이기에
내부를 보여주지 않는다.

In [ ]:
housing_cat_1hot

희소행렬은 대다수의 항목이 0인 큰 모양의 2차원 어레이를 메모리 효율적으로 다루기 위해 사용한다.
예를 들어, 훈련셋의 해안 근접도 특성을 원-핫 인코딩하면
(16512, 5) 모양의 어레이가 생성되는 데 그중에 16512개의 항목만 1이고 나머지는 0이다.
이런 경우 0이 아닌 항목의 위치 정보와 원래 행렬의 크기만 알면 되기에
희소 행렬을 대신 사용하곤 한다.

반면에 희소 행렬에 `toarray()` 메서드를 적용하면 밀집 행렬<font size='2'>dense array</font>로
변환되어 내용이 확인된다.

In [ ]:
housing_cat_1hot.toarray()

**`sparse_output=False`** 옵션

원-핫 인코딩 변화기를 지정할 때 `sparse_output=False` 옵션을 지정하면
결과가 항상 밀집 어레이로 계산된다.

In [ ]:
cat_encoder = OneHotEncoder(sparse_output=False)
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

결과적으로 원래 하나의 `ocean_proximity` 특성이 원-핫 인코딩에 의해 5개의 특성값으로 변환되었다.
예를 들어 `housing`에 포함된 0번 인덱스 구역의 `ocean_proximity`는 `NEAR BAY`이다.

In [ ]:
housing_cat.iloc[0]

`NEAR BAY`는 원-핫 인코더가 정한 특성값들의 순서상 3번 인덱스에 해당함을 `cat_encoder.categories_` 속성으로 확인된다.

In [ ]:
cat_encoder.categories_

따라서 해당 구역의 해안 근접도 특성값이 다음과 같이 길이가 5이면서
3번 인덱스의 값만 1이고 나머지는 0인 1차원 어레이로 변환되었다.

In [ ]:
housing_cat_1hot[0]

원-핫 인코딩 과정에서 새로 생성되는 특성들을 이름은 다음과 같으며,
**더미 특성**<font size='2'>dummy features</font>으로 불린다.

새롭게 생성된 5개의 특성은 기존에 주어진 특성 대신 사용되는 특성이라는 의미에서 더미 특성으로 불린다.
여기서는 해안 근접도 특성에 사용된 실제 값을 대변하는 특성으로 사용된다.

In [ ]:
cat_encoder.get_feature_names_out()

더미 특성정보를 활용하여 데이터프레임으로 다음과 같이 변환할 수 있다.

In [ ]:
pd.DataFrame(housing_cat_1hot,
             columns=cat_encoder.get_feature_names_out(),
             index=housing_cat.index)

### 특성 스케일링

**min-max 스케일링**

min-max 스케일링은 정규화의 한 방식이다.
아래 식을 이용하여 특성값 $x$를 0에서 1 사이의 값으로 변환한다.
$max$ 와 $min$ 은 각각 해당 특성값들의 최댓값과 최솟값을 가리킨다.

$$
\frac{x-min}{max-min}
$$

min-max 스케일링은 이상치에 매우 민감하다.
예를 들어 이상치가 매우 크면 분모가 분자에 비해 훨씬 크게 되어 변환된 값이 0 근처에 몰리게 된다.
사이킷런의 `MinMaxScaler` 변환기가 min-max 스케일링을 지원한다.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
housing_num_min_max_scaled = min_max_scaler.fit_transform(housing_num)

In [ ]:
housing_num_min_max_scaled

**표준화**

표준화<font size='2'>standardization</font>는 아래식을 이용하여 특성값 $x$를 변환한다.
단, $\mu$ 와 $\sigma$ 는 각각 해당 특성값들의 평균값과 표준편차를 가리킨다.

$$
\frac{x-\mu}{\sigma}
$$

표준화 스케일링으로 변환된 특성은
평균값은 0, 표준편차는 1인 분포를 따르며, 이상치에 상대적으로 덜 영향을 받는다.
사이킷런의 `StandardScaler` 변환기가 표준화 스케일링을 지원한다.

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
housing_num_std_scaled = std_scaler.fit_transform(housing_num)

In [ ]:
housing_num_std_scaled

### 로그 변환

데이터셋이 두터운 꼬리 분포를 따르는 경우,
즉 히스토그램이 지나치게 한쪽으로 편향된 경우
스케일링을 적용하기 전에 먼저
로그 함수 $\log(x)$를 적용하여 어느 정도 좌우 균형이 잡힌 분포로 변환할 것을 권장한다.
좌우 균형이 잘 잡힌 특성들을 활용하면 머신러닝 모델의 훈련이 보다 잘된다.

**`FunctionTransformer` 변환기**

로그 변화처럼 미리 어떤 정보를 확인할 필요 없이 바로 데이터 변환을 진행할 수
있다면, `fit() 메서드`를 굳이 사용할 필요가 없다.
이런 경우 `FunctionTransformer` 변환기를 활용한다.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log)
log_pop = log_transformer.transform(housing[["population"]])

In [ ]:
log_pop

아래 그림은 구역별 인구로 구성된 `population` 특성값에 로그함수를 적용할 때 분포가 보다 균형잡히는 것을 잘 보여준다.

In [ ]:
# extra code – this cell generates Figure 2–17
fig, axs = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
housing["population"].hist(ax=axs[0], bins=50)
log_pop.hist(ax=axs[1], bins=50)
axs[0].set_xlabel("Population")
axs[1].set_xlabel("Log of population")
axs[0].set_ylabel("Number of districts")

plt.show()

### 비율 변환

두 개의 특성 사이의 비율을 계산하여 다음 새로운 특성 세 개를 생성할 예정이다.

- 침실 비율: `housing['total_bedrooms'] / housing['total_rooms']`
- 가구당 방 수: `housing['total_bedrooms'] / housing['households']`
- 가구당 평균 가구원수: `housing['population'] / housing['households']`

비율 변환 또한 앞서 설명한 `FunctionTransformer` 변환기를 활용하며,
아래 코드가 간단한 사용법을 보여준다.

- `sample_array`: (2, 2) 모양의 어레이
- `ratio_transformer`: 0번 열을 1번 열로 나눈값으로 구성된 어레이로 변환하는 변환기

In [ ]:
sample_array = np.array([[1., 2.],
                         [3., 4.]])

In [ ]:
ratio_transformer = FunctionTransformer(lambda X: X[:, [0]] / X[:, [1]])
transformed_array =ratio_transformer.transform(sample_array)

In [ ]:
transformed_array

## 파이프라인

데이터 정제와 전처리의 모든 단계가 정확한 순서대로 진행되어야 한다.
사이킷런은 여러 변환기를 순차적으로 또는 병렬적으로 실행하는
파이프라인 기능을 지원한다.

사이킷런에서 제공하는 파이프라인 관련 주요 API는 다음과 같다.

| API | 설명 |
|---|---|
| `Pipeline` 클래스 | 이름으로 구분된 여러 추정기(변환기, 예측기)를 순차적으로 연결하여 파이프라인 구성. 단 예측기가 사용되는 경우 맨 마지막에 추가 |
| `make_pipeline()` 함수 | 추정기 이름 지정 없이 간편하게 `Pipeline` 객체 생성 |
| `ColumnTransformer` 클래스 | 특성마다 다른 변환기를 병렬 적용 후 결과 병합 |
| `make_column_transformer()` 함수 | 이름 지정 없이 간편하게 `ColumnTransformer` 객체 생성 |

**주의 사항**

여기서는 먼저 언급된 파이프라인 관련 주요 API 활용법을 소개한다.
그런 다음 소개된 모든 API를 조합해서 실제 캘리포니아 주택 가격 데이터셋의 변환에 사용되는 파이프라인은 지정한다.

### `Pipeline` 클래스

파이프라인으로 정의된 추정기가 변환기인지, 예측기인지 여부는
마지막 추정기에 의해 결정된다.
즉, 마지막 추정기가 변환기인지, 예측기인지에 따라
해당 파이프라인이 변환기 또는 예측기가 된다.
`num_pipeline`를 구성하는 마지막 추정기가 변환기이기에
생성된 파이프라인 역시 변환기가 된다.

파이프라인을 호출하면 마지막 추정기 이전까지의 변환기에 대해서는
`fit_transform()` 메소드가 연속적으로 호출된다.
따라서 파이프라인에 포함된 마지막 추정기를 제외한 모든 추정기는 변환기 이어야 한다.

일단 여기서는 변환기 파이프라인만 활용하며,
이후 머신러닝 모델을 구현할 때 예측기를 마지막 추정기로 지정하는
파이프라인을 이용한다.

**수치형 특성 변환 기본 파이프라인**

In [ ]:
from sklearn.pipeline import Pipeline

num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])

### `make_pipeline()` 함수

`make_pipeline()` 함수를 이용할 수도 있다.
단, 각 추정기의 이름은 자동으로 생성된다.

In [ ]:
from sklearn.pipeline import make_pipeline

num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

파이프라인 활용법은 변환기/예측기 활용법과 동일하다.

- `fit_transform()` 메서드를 호출하면 파이프라인에 포함된 변환기 각각에 대해
    `fit_transform()` 메서드가 연속적으로 호출된다.

In [ ]:
housing_num_prepared = num_pipeline.fit_transform(housing_num)
housing_num_prepared[:2].round(2)

In [ ]:
df_housing_num_prepared = pd.DataFrame(
    housing_num_prepared, columns=num_pipeline.get_feature_names_out(),
    index=housing_num.index)

In [ ]:
df_housing_num_prepared.head(2)  # extra code

**파이프라인 구조**

생성된 파이프라인의 구조를 그래프로 확인할 수 있다.

In [ ]:
from sklearn import set_config

set_config(display='diagram')

num_pipeline

생성된 파이프라인의 구조를 파이프라인 객체의 속성으로도 확인할 수 있다.

In [ ]:
num_pipeline.steps

마치 리스트처럼 인덱싱을 사용하면 파이프라인에 포함된
변환기/예측기를 하나씩 지정할 수도 있다.

In [ ]:
num_pipeline[1]

In [ ]:
num_pipeline[:-1]

변화기의 지정된 이름을 이용한 인덱싱도 가능하다.

In [ ]:
num_pipeline.named_steps

In [ ]:
num_pipeline.named_steps["simpleimputer"]

파이프라인에 포함된 변환기의 속성도 직접 지정할 수도 있다.
아래 코드는 표준화 변환기의 결측치 처리 전략을 중앙값을
사용하는 방식으로 지정한다.

In [ ]:
num_pipeline.set_params(simpleimputer__strategy="median")

### `ColumnTransformer` 클래스

특성별로 파이프라인을 지정할 수 있다.

In [ ]:
from sklearn.compose import ColumnTransformer

# 수치형 특성 리스트 지정
num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
               "total_bedrooms", "population", "households", "median_income"]
# 범주형 특성 리스트 지정
cat_attribs = ["ocean_proximity"]

# 범주형 특성 변환 파이프라인
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"))

# 전체 특성 변환기
preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

**`make_column_selector()` 함수**

파이프라인에 포함되는 각 변환기를 적용할 특성을 일일이 나열하는 일이 어려울 수 있다.
이때 지정된 자료형을 사용하는 특성들만을 선택해주는 `make_column_selector()` 함수를
유용하게 활용할 수 있다.

- `make_column_selector(dtype_include=np.number)`: 수치형 특성 모두 선택
- `make_column_selector(dtype_include=object)`: 범주형 특성 모두 선택

따라서 위 `preprocessing` 변환기를 아래와 같이 정의할 수 있다.

```python
preprocessing = ColumnTransformer([
    ("num", num_pipeline, make_column_selector(dtype_include=np.number)),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)
])
```

### `make_column_transformer` 클래스

`make_column_transformer()` 함수를 이용하여 `ColumnTransformer` 변환기를
생성할 수도 있다.
단, 각 변환기의 이름은 자동으로 생성된다.

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformer

preprocessing = make_column_transformer(
    (num_pipeline, make_column_selector(dtype_include=np.number)),
    (cat_pipeline, make_column_selector(dtype_include=object)),
)

### 캘리포니아 데이터셋 변환 파이프라인

`ColumnTransformer` 클래스와 `Pipeline` 클래스를 이용하여
캘리포니아 주택 가격 데이터의 입력 데이터셋을 한꺼번에 변환하는
파이프라인 변환기를 아래 코드에서 정의된 세 개의 파이프라인을 이용하여 구현한다.

In [ ]:
# 파이프라인 1: 두 수치형 특성의 비율을 계산하는 사용자 정의 변환기
def column_ratio(X):
    return X[:, [0]] / X[:, [1]] # 1번 특성에 대한 0번 특성의 비율율

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]  # 새로 생성되는 특성값들의 특성명

ratio_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(column_ratio, feature_names_out=ratio_name),
    StandardScaler())

# 파이프라인 2: 여러 수치형 특성에 로그 변환을 적용하는 사용자 정의 변환기
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())

# 파이프라인 3: 기타 특성(위도, 경도, 중위 주택연령 등)에 적용될 기본 수치형 특성 변환기
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())

**종합**

앞서 언급된 모든 변환기를 특성별로 적용하는 변환기를
`ColumnTransformer` 클래스를 이용하여 정의한다.

`remainder=default_num_pipeline`는
그때까지 언급되지 않은 나머지 특성들을 처리하는 변환기를
키워드 인자로 지정한다.
`remainder` 매개변수의 키워드 인자로 나머지 특성을 삭제하는 것을 지정하는
`drop` 이 기본값이며, 그 이외에 `passthrough`는 나머지 특성은
변환하지 않고 그대로 두어야 함을 의미한다.

아래 코드에서는 명시적으로 다루지 않은 나머지 특성인 `housing_median_age`, `latitude`, `longitude`에
기본 변환기(`default_num_pipeline`)를 적용하도록 지정한다.

In [ ]:
preprocessing = ColumnTransformer([
        ("bedrooms", ratio_pipeline, ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline, ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline, ["population", "households"]),
        ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                               "households", "median_income"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder=default_num_pipeline)  # 남은 특성: housing_median_age, latitude, longitude

모든 파이프라인 변환기의 사용법은 동일하다.
아래 코드는 `housing` 훈련셋을 특성별로 지정된 변환을 실행한다.
반환값이 넘파이 어레이임에 주의한다.

In [ ]:
housing_prepared = preprocessing.fit_transform(housing)

In [ ]:
housing_prepared

특성이 총 16개로 늘었으며,
특성별 이름은 변환기의 `get_feature_names_out()` 메서드로 확인한다.

- 비율 변환기 적용: 3개의 새로운 특성 추가.
- 해안 근접도 더미 특성: 5개로 변환. 기존 해안 근접도 특성 제거 후 5개 더미 특성 추가.
- 나머지 특성: 3개(중위 주택연령, 위도, 경도) 변환. 특성수는 그대로 유지됨(표준화만 적용)

In [ ]:
housing_prepared.shape

In [ ]:
preprocessing.get_feature_names_out()

`preprocessing`에 의한 변환 과정에서 새로 생성된 특성 이름과 함께 확인하면 다음과 같다.

In [ ]:
housing_prepared_df = pd.DataFrame(housing_prepared,
                                   columns=preprocessing.get_feature_names_out(),
                                   index=housing.index)
housing_prepared_df.head()

In [ ]:
housing_prepared_df.info()

## 모델 선택과 훈련

`preprocessing`에 의해 변환되는 데이터프레임은 예측기 모델의 훈련에
바로 사용될 수 있다.
즉, 이제 머신러닝 예측기 모델의 훈련셋으로 바로 사용할 수 있다.

하지만 여기서는 데이터 변환과 모델 훈련을 분리해서 진행하는 대신
변환기와 예측기를 하나의 파이프라인으로 묶어
데이터 변환과 모델 훈련을 동시에 진행하는 방법을 선택해서 소개한다.

`preprocessing`이 가리키는 변환기와 함께 묶여 하나의 파이프라인으로 구성될 예측기로
사이킷런의 회귀 모델 세 개를 활용한다.
각 모델의 자세한 특징과 상세 설명은 이어지는 장에서 하나씩 소개할 예정이며,
여기서는 모델 선택에 따른 성능과 보다 좋은 모델을 훈련시키는 방법을 자세히 소개한다.

### 모델 훈련과 평가

**선형회귀 모델 활용**

- 훈련

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)

- 예측

In [ ]:
housing_predictions = lin_reg.predict(housing)
housing_predictions[:5].round(-2)  # -2 = rounded to the nearest hundred

실제 주택 중위가격은 다음과 같으며, 예측값과 오차가 꽤 난다.

In [ ]:
housing_labels.iloc[:5].values

In [ ]:
error_ratios = housing_predictions[:5].round(-2) / housing_labels.iloc[:5].values - 1
print(", ".join([f"{100 * ratio:.1f}%" for ratio in error_ratios]))

- 훈련셋에 대한 RMSE

예측값의 RMSE가 매우 높게 나온다.
모델 훈련이 제대로 진행되지 못한 과소 적합이 발생하였으며 이는 선형회귀 모델이 적절하지 않음을 의미한다.

In [ ]:
from sklearn.metrics import root_mean_squared_error

lin_rmse = root_mean_squared_error(housing_labels, housing_predictions)
lin_rmse

**결정트리 회귀 모델 활용**

- 훈련

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels)

- 훈련셋에 대한 RMSE

RMSE가 0으로 나온다.
이는 결정트리 모델이 심하게 과대 적합되었음을 의미한다.
이런 모델은 전혀 의미가 없다.

In [ ]:
housing_predictions = tree_reg.predict(housing)
tree_rmse = root_mean_squared_error(housing_labels, housing_predictions)
tree_rmse

**랜덤 포레스트 회귀 모델**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing,
                           RandomForestRegressor(n_estimators=100, random_state=42))

- 훈련

In [ ]:
forest_reg.fit(housing, housing_labels)

- 훈련셋에 대한 RMSE

RMSE가 결정트리보다는 당연히 높지만 선형회귀 모델 보다는 훨씬 낮다.

In [ ]:
housing_predictions = forest_reg.predict(housing)
forest_rmse = root_mean_squared_error(housing_labels, housing_predictions)
forest_rmse

### 교차 검증

**사이킷런의 `cross_val_score()` 함수**

`cross_val_score()` 함수는 지정된 모델을 k-겹 교차 검증을 활용하여 훈련과 평가를 동시에 진행한다.
교차검증은 다만 모델 평가용도로만 폴드를 구분하여 훈련할 뿐 훈련된 모델 객체 자체를 반환하지는 않는다.
예를 들어 아래 코드는 결정트리 모델에 대해 교차 검증을 실행한다.

`cross_val_score()` 함수 호출에 사용된 키워드 인자는 다음과 같다.

- `scoring="neg_mean_squared_error"` 옵션
    - 훈련중인 모델의 성능을 측정하는 **효용함수** 지정
    - 모델의 성능 측정값은 높을 수록 좋은 성능으로 평가되기에 회귀 모델의 경우 일반적으로 RMSE의 음숫값을 사용함.
- `cv=5`: 5-겹 교차 검증 진행

`cross_val_score()` 함수의 반환값은 `scoring="neg_mean_squared_error"` 옵션으로 인해 음수값이다.
따라서 다시 양수로 만들어서 `tree_rmses` 변수에 할당한다.

**결정트리 모델에 대한 교차 검증**

In [ ]:
from sklearn.model_selection import cross_val_score

tree_rmses = -cross_val_score(tree_reg, housing, housing_labels,
                              scoring="neg_root_mean_squared_error", cv=5)

`cv=5` 설정에 의해 5개의 폴드를 사용하며 매번 RMSE를 측정한다.

In [ ]:
tree_rmses

`pandas.Series` 로 변환하면 통계 정보를 쉽게 구할 수 있다. 결과가 이전에 하나의 결정트리 모델만 사용했을 때 보다 훨씬 나쁘다.

In [ ]:
pd.Series(tree_rmses).describe()

**선형회귀 모델에 대한 교차 검증**

In [ ]:
lin_rmses = -cross_val_score(lin_reg, housing, housing_labels,
                              scoring="neg_root_mean_squared_error", cv=5)
pd.Series(lin_rmses).describe()

**랜덤 포레스트 모델에 대한 교차 검증**

램덤 포레스트 모델에 교차 검증을 적용하면 보다 많은 시간이 걸린다 (몇 분 정도).
그만큼 랜덤 포레스트 모델이 보다 복잡한 훈련을 진행하기 때문이다.
하지만 결과적으로 랜덤 포레스트 회귀 모델의 성능이 보다 좋다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing,
                           RandomForestRegressor(random_state=42))
forest_rmses = -cross_val_score(forest_reg, housing, housing_labels,
                                scoring="neg_root_mean_squared_error", cv=5)

In [ ]:
pd.Series(forest_rmses).describe()

훈련셋에 대한 RMSE에 비하면 매우 높아졌다. 따라서 랜덤 포레스트 모델 또한 훈련셋에 너무 특화되어 있다, 즉 과대적합이 발생하였다.

## 모델 미세 조정

### 그리드 탐색

그리드 탐색에 사용될 모델을 전처리와 함께 지정한다.
파이프라인에 포함된 전처리와 예측기에 사용되는 하이퍼파라미터 중에서 미세조정에 사용될 하이퍼파라미터가 가질 수 있는 값들의 리스트를 지정한다.

랜덤 포레스트 모델의 `n_estimators`(트리 개수)와 `max_features`(분할에 사용할 최대 특성 수)
두 하이퍼파라미터를 대상으로 총 15개의 조합에 대해 모델을 지정한 다음에
매번 3-겹 교차 검증을 실행하기에
아래 코드는 총 45번 훈련을 진행한다.

```
(3 * 3 + 2 * 3) * 3 = 45
```

아래 코드를 실행하면 컴퓨터 사양에 따라 몇 분 이상 걸릴 수 있다.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 랜덤 포레스트 모델과 전체 전처리 파이프라인을 하나로 묶은 파이프라인
full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])

# 하이퍼파라미터 조합: 3*3 + 2*3 조합 확인
# 랜덤 포레스트 모델의 트리 개수(n_estimators)와 최대 특성 수(max_features) 대상 지정
param_grid = [
    {'random_forest__n_estimators': [50, 100, 150],   # 트리 개수
     'random_forest__max_features': [4, 6, 8]},        # 최대 특성 수
    {'random_forest__n_estimators': [100, 200],
     'random_forest__max_features': [8, 10, 12]},
]

# 그리드 탐색 객체 생성: 전체 파이프라인과 하이퍼파라미터 그리드, 교차 검증 분할 수, 평가 지표 지정
grid_search = GridSearchCV(full_pipeline, param_grid, cv=3,
                           scoring='neg_root_mean_squared_error')

# 그리드 탐색 수행
grid_search.fit(housing, housing_labels)

**`full_pipeline.get_params().keys()`**

파이프라인에 포함된 변환기와 예측기의 하이퍼파라미터는 각 객체의 `get_params()` 메서드를 호출해서
각 객체별로 확인하거나,
파이프라인 자체의 `get_params()` 메서드를 호출하여 파이프라인 객체를
지정할 때 사용할 수 있는 하이퍼파라미터 전체를 확인할 수 있다.
예를 들어, 사용가능한 10개의 하이퍼파라미터는 다음과 같다.

In [ ]:
(list(full_pipeline.get_params().keys()))[:10]

하지만 개별 객체의 하이퍼파라미터를 사이킷런 문서에서 하나씩 확인해서
모델 미세 조정에 사용하면 좋을 중요한 하이퍼파라미터를 지정할 것을 권장한다.
어떤 하이퍼파라미터가 중요한가는 앞으로 하나씩 알아나갈 것이다.

**그리드 탐색 결과**

- `grid_search.best_params_` 속성

그리드 탐색을 통해 찾아낸 최적의 하이퍼파라미터 조합은 다음과 같다.

In [ ]:
grid_search.best_params_

- `grid_search.best_estimator_` 속성

그리드 탐색을 통해 찾아낸 최적의 모델은 다음과 같다.

In [ ]:
grid_search.best_estimator_

- `grid_search.cv_results_` 속성

그리드 탐색 과정에서 훈련된 15개 모델 각각의 평가지표를 확인할 수 있다.
원래 사전 자료형으로 지정되며,
키(key)는 랜덤 포레스트 모델의 하이퍼파라미터와 각 모델의 훈련 성능과 관련된다.

사전을 데이터프레임으로 변환하면 보다 보기가 편하다.
데이터프레임의 열은 사전 자료형의 키를 사용한다.

In [ ]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True) # 평균 테스트 점수 기준으로 결과 정렬

# 일부 특성 및 점수 열만 선택하여 보기 좋게 정리
cv_res = cv_res[["param_random_forest__n_estimators",
                 "param_random_forest__max_features", "split0_test_score",
                 "split1_test_score", "split2_test_score", "mean_test_score"]]
score_cols = ["split0", "split1", "split2", "mean_test_rmse"]

cv_res.columns = ["n_estimators", "max_features"] + score_cols
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)

# 상위 성능 5개 조합 확인
cv_res.head()

### 랜덤 탐색

그리드 탐색은 적은 수의 하이퍼파라미터 조합을 실험해볼 때만 유용하다.
반면에 하이퍼파라미터 탐색 공간이 커서 조합 경우의 수가 많아지면 훈련 시간이 너무
올래 걸려 활용하기 어렵다.
이런 경우 랜덤 탐색<font size='2'>randomized search</font>이 보다 효율적으로 최적의 하이퍼파라미터 조합을 찾아낼 수 있다.

아래 코드는 무작위로 선택한 10개의 하이퍼파라미터 조합에 대해 3-겹 교차 검증을 진행하기에 총 30(=10x3)번 훈련을 진행한다.
아래 코드는 컴퓨터 사양에 따라 몇 분 이상 걸릴 수 있다.

**무작위 추출 표본분포 함수 선택**

랜덤 탐색을 진행하려면
지정된 하이퍼파라미터에 대한 값을 무작위로 지정하는 데에 사용되는
표본분포 함수를 지정해야 한다.
아래 랜덤 탐색에서는 이산 균등 분포를 사용하는 `randint()` 함수를 이용한다.

```python
'random_forest__n_estimators': randint(low=50, high=300)
'random_forest__max_features': randint(low=2, high=20)
```

`scipy` 라이브러리는 이외에 다른 종류의 확률 분포 함수를 지원한다.
예를 들어, 지정된 구간에서의 부동소수점을 선택해야 한다면
연속 균등분포 함수인 `scipy.stats.uniform(a, b)`를 이용할 수 있다.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 랜덤 탐색을 위한 하이퍼파라미터 분포 지정:
# - 트리 개수는 50~300 범위에서 균등 분포로 샘플링
# - 최대 특성 수는 2~20 범위에서 균등 분포로 샘플링
param_distribs = {'random_forest__n_estimators': randint(low=50, high=300),
                  'random_forest__max_features': randint(low=2, high=20)}

# 랜덤 탐색 객체 생성: 전체 파이프라인과 하이퍼파라미터 분포, 교차 검증 분할 수, 평가 지표 지정
rnd_search = RandomizedSearchCV(
    full_pipeline, param_distributions=param_distribs, n_iter=10, cv=3,
    scoring='neg_root_mean_squared_error', random_state=42)

# 랜덤 탐색 수행
rnd_search.fit(housing, housing_labels)

랜덤 탐색 과정에서 훈련된 10개 모델 각각의 하이퍼파라미터와
최고 성능 모델 등의 정보는 그리드 탐색 객체와 동일한 속성에 저장된다.

In [ ]:
rnd_search.best_params_

In [ ]:
cv_res = pd.DataFrame(rnd_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)

cv_res = cv_res[["param_random_forest__n_estimators",
                 "param_random_forest__max_features", "split0_test_score",
                 "split1_test_score", "split2_test_score", "mean_test_score"]]

cv_res.columns = ["n_estimators", "max_features"] + score_cols
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)

cv_res.head()

**하이퍼파라미터를 위한 샘플링 분포 선택 방법**

* `scipy.stats.randint(a, b+1)`
    - a부터 b까지의 범위를 갖는 **이산형(정수형)** 하이퍼파라미터에 사용
    - 해당 범위 내의 모든 값이 선택될 확률이 동일하다고 예상될 때 활용
* `scipy.stats.uniform(a, b)`
    - `randint()` 함수와 매우 유사
    - 하지만 **연속형(실수형)** 하이퍼파라미터에 사용

아래는 `randint()`와 `uniform()`에 대한 확률 질량 함수(이산 확률 변수용)와 확률 밀도 함수(연속 확률 변수용)를 나타낸 그래프이다.

In [ ]:
from scipy.stats import randint, uniform, geom, expon

xs1 = np.arange(0, 7 + 1)
randint_distrib = randint(0, 7 + 1).pmf(xs1)

xs2 = np.linspace(0, 7, 500)
uniform_distrib = uniform(0, 7).pdf(xs2)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.bar(xs1, randint_distrib, label="scipy.randint(0, 7 + 1)")
plt.ylabel("Probability")
plt.legend()
plt.axis([-1, 8, 0, 0.2])

plt.subplot(1, 2, 2)
plt.fill_between(xs2, uniform_distrib, label="scipy.uniform(0, 7)")
plt.ylabel("PDF")
plt.legend()
plt.axis([-1, 8, 0, 0.2])

plt.show()

## 최적 모델 활용 및 평가

랜덤 탐색으로 확인된 최적의 모델을 활용하여 최종 평가를 진행한다.

In [ ]:
final_model = rnd_search.best_estimator_  # includes preprocessing

### 테스트셋 활용

이제 더 이상의 훈련을 진행할 필요가 없을 정도로 훈련된 모델의 성능에
만족한다고 가정하자.
다음 할 일은 모델을 실전에 투입했을 때의 모델 성능 예측하기이며,
여기에 테스트셋을 활용한다.
테스트셋 또한 입력 데이터셋과 타깃셋으로 먼저 구분되어야 하고,
모델 자체에 전처리 기능이 포함되어 있음에 주의한다.

In [ ]:
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

final_predictions = final_model.predict(X_test)

final_rmse = root_mean_squared_error(y_test, final_predictions)
print(final_rmse)

SciPy의 `bootstrap()` 함수를 사용하여 테스트 RMSE에 대한
95% 신뢰구간<font size='2'>confidence interval</font>를 계산하면 다음과 같다.
신뢰구간은 줄여서 CI라 한다.

In [ ]:
from scipy.stats import bootstrap

def rmse(squared_errors):
    return np.sqrt(np.mean(squared_errors))

confidence = 0.95
squared_errors = (final_predictions - y_test) ** 2
boot_result = bootstrap([squared_errors], rmse, confidence_level=confidence,
                        random_state=42)
rmse_lower, rmse_upper = boot_result.confidence_interval
print(f"95% CI for RMSE: ({rmse_lower:.4f}, {rmse_upper:.4f})")

### 학습된 모델 기타 활용법

머신러닝 모델은 단순히 예측을 위해서만 사용되지는 않는다.
모델 종류에 따라 예측값 계산과 함께 다른 기능을 제공하기도 한다.

예를 들어, 훈련이 잘 진행된 랜덤 포레스트 모델은
입력 데이터셋의 각 특성이 모델이 예측값을 계산할 때 얼마나 많이 기여하는가를
특성 중요도라는 기준으로 훈련 과정중에 평가한다.
`feature_importances_` 속성에 특성별 중요도가 저장되며, 아래 코드로 중요도가 높은 순으로 확인한다.

In [ ]:
# 모델에 포함된 랜덤 포레스트 모델을 이용한 특성 중요도 추출
feature_importances = final_model["random_forest"].feature_importances_

# 특성 중요도와 해당 특성명을 함께 묶어서 중요도 기준으로 정렬
important_features = sorted(zip(final_model["preprocessing"].get_feature_names_out(),
                                feature_importances),
                            key=lambda x: x[1],
                            reverse=True)
important_features[:10]

### 모델 저장

최적의 모델을 훈련시키는 과정이 매우 길 수 있다.
따라서 한 번 훈련된 좋은 모델은 파일로 저장해 놓아야 한다.
그러면 모델을 활용하고자 할 때 저장된 파일을 모델로 불러와서
훈련 없이 바로 활용할 수 있다.
또한 새롭게 훈련시킨 모델이 적절하지 않다고 판단되어
이전 버전의 모델로 되돌려야 하는 상황이 발생할 수도 있기에
잘 훈련된 모델의 저장은 필수적이다.

모델의 저장과 불러오기는 각각 `joblib` 모듈의
`dump()` 함수와 `load()` 함수를  활용한다.

- 저장하기

    ```python
    import joblib
    joblib.dump(final_model, "my_california_housing_model.pkl")
    ```
- 불러오기와 활용

    ```python
    final_model_reloaded = joblib.load("my_california_housing_model.pkl")
    final_model_reloaded.predict(X_test)
    ```

**모델 저장**

아래 코드를 실행하면 지정된 경로에 최적의 모델을 pickle 파일로 저장한다.

In [ ]:
import joblib

joblib.dump(final_model, "my_california_housing_model.pkl")

저장된 모델을 다시 불러와 바로 실전에 투입할 수 있다.
다만, 저장된 모델 활용에 필요한 라이브러리(함수) 정의를 함께 해야 한다.
그렇지 않으면 모델 불러오고 활용할 때 필요한 클래스나 함수가 정의되지 않아 오류가 발생한다.

In [ ]:
import joblib

### 중요 안내 시작 ###
# 저장된 모델 활용에 필요한 함수 정의를 함께 해야 함.
# 그렇지 않으면 모델 로드 시 필요한 클래스나 함수가 정의되지 않아 오류 발생
# (이 노트북에서는 이미 위에서 정의했으므로 생략)

# def column_ratio(X):
#     return X[:, [0]] / X[:, [1]]
### 중요 안내 종료 ###

# 모델 불러오기
final_model_reloaded = joblib.load("my_california_housing_model.pkl")

# 새로운 데이터에 대한 예측에 활용 예제
new_data = housing.iloc[:5]  # 새로운 데이터라고 가정
predictions = final_model_reloaded.predict(new_data)
print(predictions)

---

## 선형회귀 모델은 내부에서 어떻게 학습되나

지금까지는 `LinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor` 등을
**블랙박스**로 취급하여 `fit()`과 `predict()`만 호출해서 사용했다.
지금부터는 그 중 가장 기본이 되는 **선형회귀 모델이 내부에서 어떻게 학습되는지**를
경사하강법 실험을 통해 직접 들여다본다.
이어서 선형회귀를 확장한 **다항 회귀**, 과대적합을 억제하는 **모델 규제**,
그리고 훈련을 적절한 시점에 멈추는 **조기 종료**까지 사이킷런 코드로 직접 실습한다.

## 선형 회귀 배치 학습 실험

**실험용 학습 데이터 생성 및 시각화**

배치 경사하강법 실험을 위한 훈련 데이터를 생성한다.
실제 관계는

$$y = 4 + 3\, x_1 + \text{noise}$$

이며, 0에서 2 사이의 구간에서 200개의 샘플을 균등하게 무작위로 선택한다.
즉, $x_1 \sim U(0, 2)$이 성립한다.
$y$ 값에 추가된 노이즈 또한 0에서 1 사이의 구간에서 균등하게 무작위로 선택된다.

아래 코드는 데이터 생성과 생성된 데이터를 산점도를 그린다.

In [ ]:
rng = np.random.default_rng(seed=42)  # 재현 가능한 난수 생성기 (시드 고정)
m = 200  # 훈련 샘플 수

# 입력 특성: [0, 2) 구간에서 균등 분포로 200개 샘플 생성 → (200, 1) 형태
X = 2 * rng.random((m, 1))

# 타깃값: y = 4 + 3*x1 + 가우시안 잡음 (평균 0, 표준편차 1)
y = 4 + 3 * X + rng.standard_normal((m, 1))

# 산점도 시각화
plt.figure(figsize=(6, 4))
plt.plot(X, y, "b.")         # 파란 점으로 데이터 표시
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([0, 2, 0, 15])     # x축 구간: [0, 2], y축 구간: [0, 15]
plt.grid()

plt.show()

**배치 경사하강법 시각화 함수 정의**

`plot_gradient_descent(theta, eta)` 함수를 정의한다.
이 함수는 초기 파라미터 `theta`와 학습률 `eta`를 입력받아 1000 에포크 동안 배치 경사하강법을 실행한다.
처음 20 에포크 동안 매 에포크마다 훈련된 선형 회귀 모델의 예측값을 직선으로 그려
학습된 편향과 절편의 변화를 시각적으로 보여준다.

각 에포크에서의 비용 함수의 그래디언트는 아래 수식으로 계산되며,
아래 코드에서 `gradients` 변수가 이를 가리킨다.

$$\nabla_\theta \text{MSE} = \frac{2}{m} X^T (X\theta - y)$$

In [ ]:
import matplotlib as mpl
from sklearn.preprocessing import add_dummy_feature

# 훈련 데이터에 편향 항(x0 = 1) 추가 → (200, 2) 형태: [1, x1]
X_b = add_dummy_feature(X)

# 예측선 시각화를 위한 새로운 입력값: x1 = 0과 x1 = 2 두 점
X_new = np.array([[0], [2]])
X_new_b = add_dummy_feature(X_new)  # 편향 항 추가 → [[1, 0], [1, 2]]

def plot_gradient_descent(theta, eta):
    """
    배치 경사하강법을 실행하고 수렴 과정을 시각화한다.
    - theta: 초기 파라미터 (편향 θ0, 기울기 θ1)
    - eta: 학습률
    """
    m = len(X_b)           # 훈련 샘플 수
    plt.plot(X, y, "b.")   # 훈련 데이터 산점도
    n_epochs = 1000        # 총 에포크 수
    n_shown = 20           # 처음 n_shown 에포크의 예측선만 시각화
    theta_path = []        # 에포크별 파라미터 저장 리스트

    for epoch in range(n_epochs):
        if epoch < n_shown:
            # 현재 theta로 예측선 계산 후 시각화 (에포크 순서에 따라 색상 변화)
            y_predict = X_new_b @ theta
            color = mpl.colors.rgb2hex(plt.cm.OrRd(epoch / n_shown + 0.15))
            plt.plot(X_new, y_predict, linestyle="solid", color=color)

        # 배치 경사하강법: 전체 훈련 데이터로 그래디언트 계산
        # ∇MSE = (2/m) * X_b^T @ (X_b @ theta - y)
        gradients = 2 / m * X_b.T @ (X_b @ theta - y)

        # 파라미터 업데이트: θ ← θ - η * ∇MSE
        theta = theta - eta * gradients
        theta_path.append(theta)   # 업데이트된 파라미터 저장

    plt.xlabel("$x_1$")
    plt.axis([0, 2, 0, 15])
    plt.grid()
    plt.title(fr"$\eta = {eta}$")  # 서브플롯 제목에 학습률 표시
    return theta_path

**학습률에 따른 배치 경사하강법 수렴 비교**

세 가지 학습률($\eta$)로 배치 경사하강법을 실행하여 결과를 비교한다.

- $\eta = 0.02$ (너무 작음): 스텝 크기가 작아 수렴이 매우 느리고, 1000 에포크 안에 최적해에 도달하지 못할 수 있다.
- $\eta = 0.1$ (적절함): 빠르고 안정적으로 최적해에 수렴한다.
- $\eta = 0.5$ (너무 큼): 스텝이 지나치게 커서 최솟값을 건너뛰며 발산하거나 진동한다.

In [ ]:
rng = np.random.default_rng(seed=42)
theta = rng.standard_normal((2, 1))  # 파라미터 무작위 초기화: [θ0(편향), θ1(기울기)]

plt.figure(figsize=(10, 4))

# 왼쪽 서브플롯: η = 0.02 (학습률 너무 작음 → 느린 수렴)
plt.subplot(131)
plot_gradient_descent(theta, eta=0.02)
plt.ylabel("$y$", rotation=0)

# 가운데 서브플롯: η = 0.1 (적절한 학습률 → 빠르고 안정적인 수렴)
plt.subplot(132)
plot_gradient_descent(theta, eta=0.1)
plt.gca().axes.yaxis.set_ticklabels([])  # y축 눈금 레이블 숨김

# 오른쪽 서브플롯: η = 0.5 (학습률 너무 큼 → 발산 또는 진동)
plt.subplot(133)
plot_gradient_descent(theta, eta=0.5)
plt.gca().axes.yaxis.set_ticklabels([])  # y축 눈금 레이블 숨김

plt.show()

## 다항 회귀

지금까지 사용한 데이터는 직선 관계($y = 4 + 3x_1$)를 따랐다.
하지만 현실의 데이터는 직선이 아닌 **곡선** 형태의 관계를 갖는 경우가 많다.
이런 비선형 데이터를 선형회귀 모델로 학습하는 대표적인 방법이 **다항 회귀**<font size='2'>polynomial regression</font>이다.

아이디어는 간단하다 — 원래 특성 $x_1$의 거듭제곱($x_1^2$, $x_1^3$, ...)을
새로운 특성으로 추가한 다음, 확장된 특성에 대해 **선형회귀**를 적용한다.
사이킷런의 `PolynomialFeatures` 변환기가 이 특성 확장을 지원한다.

아래 코드는

$$y = 0.5\, x_1^2 + x_1 + 2 + \text{noise}$$

관계를 따르는 비선형 데이터를 생성한다.

In [ ]:
rng = np.random.default_rng(seed=42)
m = 100
X_quad = 6 * rng.random((m, 1)) - 3           # [-3, 3) 구간의 100개 샘플
y_quad = 0.5 * X_quad ** 2 + X_quad + 2 + rng.standard_normal((m, 1))

plt.figure(figsize=(6, 4))
plt.plot(X_quad, y_quad, "b.")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.grid()
plt.show()

`PolynomialFeatures(degree=2)` 변환기는 $x_1$ 하나의 특성을
$x_1$과 $x_1^2$ 두 개의 특성으로 확장한다.
확장된 특성에 사이킷런의 `LinearRegression` 모델을 그대로 적용하면 된다.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_quad_poly = poly_features.fit_transform(X_quad)

# 원본 특성 x1과 새로 추가된 x1^2 특성을 함께 확인
X_quad[0], X_quad_poly[0]

In [ ]:
poly_lin_reg = LinearRegression()
poly_lin_reg.fit(X_quad_poly, y_quad)

# 학습된 절편과 계수: 실제 관계식(0.5, 1, 2)과 비슷하게 학습되었는지 확인
poly_lin_reg.intercept_, poly_lin_reg.coef_

학습된 모델이 실제로 곡선을 잘 따라가는지 시각화해서 확인한다.
비교를 위해 **1차(직선, 과소적합)**, **2차(적절)**, **고차(300차, 과대적합)** 세 모델을 함께 그려본다.

In [ ]:
X_new = np.linspace(-3, 3, 100).reshape(100, 1)

plt.figure(figsize=(7, 5))

for degree, style, width in [(1, "g-", 2), (2, "b-", 2), (300, "r-", 1)]:
    polybig_features = PolynomialFeatures(degree=degree, include_bias=False)
    std_scaler = StandardScaler()
    lin_reg = LinearRegression()
    polynomial_regression = make_pipeline(polybig_features, std_scaler, lin_reg)
    polynomial_regression.fit(X_quad, y_quad)
    y_newbig = polynomial_regression.predict(X_new)
    label = f"degree={degree}"
    plt.plot(X_new, y_newbig, style, label=label, linewidth=width)

plt.plot(X_quad, y_quad, "b.", alpha=0.3)
plt.legend()
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([-3, 3, -2, 12])
plt.grid()
plt.show()

- **1차(초록)**: 직선이라서 곡선 패턴을 전혀 반영하지 못한다 → **과소적합**
- **2차(파랑)**: 실제 관계식과 거의 일치하는 부드러운 곡선을 그린다 → **적절한 모델**
- **300차(빨강)**: 훈련 데이터 하나하나에 지나치게 민감하게 반응하며 요동친다 → **과대적합**

차수(degree)가 높아질수록 모델의 표현력은 커지지만, 그만큼 훈련 데이터의 잡음까지
암기하듯 학습해버릴 위험, 즉 과대적합의 위험도 커진다.

## 모델 규제

과대적합을 억제하는 대표적인 방법이 **모델 규제**<font size='2'>regularization</font>다.
모델이 훈련 데이터에 너무 민감하게(즉, 계수가 너무 커지도록) 학습되는 것을 막기 위해
비용함수에 계수 크기에 대한 페널티 항을 추가한다.

- **Ridge(L2 규제)**: 계수들의 **제곱합**에 비례한 페널티 추가 → 계수를 고르게 작게 축소
- **Lasso(L1 규제)**: 계수들의 **절댓값 합**에 비례한 페널티 추가 → 일부 계수를 아예 0으로 만들어 특성 선택 효과
- 두 경우 모두 규제 강도를 결정하는 하이퍼파라미터 `alpha`가 클수록 규제가 강해져 모델이 더 단순(평평)해진다.

아래 코드는 앞서 사용한 2차 곡선 데이터에 다양한 `alpha` 값으로 Ridge 회귀를 적용하여
규제 강도에 따라 예측 곡선이 어떻게 달라지는지 비교한다.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

def plot_regularized_model(model_class, X, y, alphas, degree=10, **model_kwargs):
    """주어진 alpha 값들에 대해 규제 모델을 훈련하고 예측 곡선을 함께 그린다."""
    X_new = np.linspace(-3, 3, 100).reshape(100, 1)
    plt.plot(X, y, "b.", alpha=0.3)

    for alpha, style in zip(alphas, ["r-", "g--", "b:"]):
        model = model_class(alpha, **model_kwargs) if alpha > 0 else LinearRegression()
        regularized = make_pipeline(
            PolynomialFeatures(degree=degree, include_bias=False),
            StandardScaler(),
            model)
        regularized.fit(X, y)
        y_new = regularized.predict(X_new)
        plt.plot(X_new, y_new, style, linewidth=2, label=fr"$\alpha = {alpha}$")

    plt.xlabel("$x_1$")
    plt.axis([-3, 3, -2, 12])
    plt.legend()
    plt.grid()

plt.figure(figsize=(12, 4))

plt.subplot(121)
plot_regularized_model(Ridge, X_quad, y_quad, alphas=(0, 1, 100))
plt.ylabel("$y$", rotation=0)
plt.title("Ridge (L2)")

plt.subplot(122)
plot_regularized_model(Lasso, X_quad, y_quad, alphas=(0, 0.01, 1), max_iter=10_000)
plt.title("Lasso (L1)")

plt.show()

`alpha = 0`(빨간 실선)은 규제가 전혀 없는 순수한 다항 회귀로,
10차 다항 특성을 모두 사용해 데이터의 잡음까지 따라가며 구불거린다 → **과대적합**.
`alpha`를 키울수록(초록 점선 → 파란 점선) 곡선이 점점 평평해지며 실제 2차 곡선에 가까워진다.
다만 `alpha`가 지나치게 크면 반대로 데이터의 패턴 자체를 놓치는 **과소적합**이 발생할 수 있으므로,
적절한 `alpha`를 찾는 과정(예: 교차 검증)이 필요하다.

## 조기 종료

**조기 종료**<font size='2'>early stopping</font>는 검증셋에 대한 손실(오차)이
더 이상 개선되지 않는 순간 훈련을 멈추는 규제 기법이다.
에포크를 거듭할수록 모델이 훈련셋에 점점 더 특화되어(과대적합되어) 가는데,
검증 오차를 매 에포크 확인하다가 오차가 더 이상 줄지 않고 최저점을 지난 시점의 모델을 선택하면
과대적합이 심해지기 전 단계의 모델을 확보할 수 있다.

아래 실습에서는 앞서 사용한 2차 곡선 데이터를 20차 다항 특성으로 크게 확장한 뒤
(일부러 과대적합이 잘 일어나도록) 규제 없는 `SGDRegressor`로 한 번에 한 에포크씩 훈련시키면서
매 에포크 검증셋 RMSE를 기록한다.

In [ ]:
from sklearn.model_selection import train_test_split

# 훈련셋을 다시 훈련용/검증용으로 분리 (검증셋은 조기 종료 시점 판단에 사용)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_quad, y_quad.ravel(), test_size=0.2, random_state=42)

20차 다항 특성으로 확장한 다음 표준화한다.
`fit_transform()`은 훈련셋에서만, `transform()`은 검증셋에 적용해야
검증셋 정보가 훈련 과정에 유출되지 않는다.

In [ ]:
poly_scaler = make_pipeline(
    PolynomialFeatures(degree=20, include_bias=False),
    StandardScaler())

X_train_prep = poly_scaler.fit_transform(X_train)
X_valid_prep = poly_scaler.transform(X_valid)

`SGDRegressor`를 `warm_start=True`로 생성하면 `fit()`을 다시 호출해도
파라미터를 초기화하지 않고 이전 상태에서 이어서 훈련한다.
`penalty=None`으로 지정하여 규제를 전혀 사용하지 않았음에 주목한다 —
오직 **조기 종료만으로** 과대적합을 억제하는 효과를 관찰하기 위함이다.

매 에포크마다 `partial_fit()`으로 한 번만 훈련을 진행한 후 검증셋 RMSE를 측정하고,
지금까지 관찰된 것 중 가장 낮은 검증 RMSE를 기록한 모델을 `best_model`로 저장해 둔다.

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import root_mean_squared_error
from copy import deepcopy

sgd_reg = SGDRegressor(penalty=None, eta0=0.002, random_state=42,
                       max_iter=1, tol=None, warm_start=True,
                       learning_rate="constant")

n_epochs = 500
best_valid_rmse = float("inf")
best_epoch = None
best_model = None
train_errors, valid_errors = [], []

for epoch in range(n_epochs):
    sgd_reg.partial_fit(X_train_prep, y_train)  # 한 에포크만 이어서 훈련

    y_valid_predict = sgd_reg.predict(X_valid_prep)
    valid_error = root_mean_squared_error(y_valid, y_valid_predict)
    valid_errors.append(valid_error)

    y_train_predict = sgd_reg.predict(X_train_prep)
    train_errors.append(root_mean_squared_error(y_train, y_train_predict))

    if valid_error < best_valid_rmse:      # 지금까지 최고 성능 갱신
        best_valid_rmse = valid_error
        best_epoch = epoch
        best_model = deepcopy(sgd_reg)      # 최고 성능 모델을 별도로 저장

print(f"조기 종료 지점: {best_epoch} 에포크, 검증 RMSE = {best_valid_rmse:.3f}")
print(f"마지막(500) 에포크의 검증 RMSE = {valid_errors[-1]:.3f}")

훈련셋과 검증셋의 RMSE 변화를 함께 그려서 조기 종료 지점을 시각적으로 확인한다.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_errors, "b--", linewidth=2, label="train RMSE")
plt.plot(valid_errors, "r-", linewidth=2, label="valid RMSE")
plt.axvline(x=best_epoch, color="k", linestyle=":", linewidth=2,
           label=f"early stopping (epoch={best_epoch})")
plt.scatter([best_epoch], [best_valid_rmse], c="k", s=80, zorder=5)

plt.xlabel("epoch")
plt.ylabel("RMSE")
plt.axis([0, n_epochs, 0, 2.5])
plt.legend()
plt.grid()
plt.show()

훈련셋 RMSE(파란 점선)는 에포크가 지날수록 계속 낮아지는 반면,
검증셋 RMSE(빨간 실선)는 어느 지점 이후로는 더 이상 낮아지지 않고 미세하게 출렁인다.
`best_model`(조기 종료 지점의 모델)을 최종 모델로 선택하면,
끝까지 훈련을 계속한 모델보다 **검증셋 기준으로 더 좋은(또는 동등한) 일반화 성능**을 가진 모델을 얻을 수 있다.

정리하면, 다항 회귀·모델 규제·조기 종료는 서로 다른 방식으로 같은 문제,
즉 **모델이 훈련 데이터에 너무 딱 맞춰져서(과대적합) 새로운 데이터에는 오히려 잘 안 맞는 상황**을
예방한다는 공통된 목표를 가진다.

---

## 정리

이번 실습에서는 캘리포니아 주택가격 데이터를 이용해 머신러닝 프로젝트의 전체 흐름
(데이터 탐색 → 전처리 → 파이프라인 → 모델 선택 → 튜닝 → 평가)을 경험했고,
그 핵심에 있는 **선형회귀 모델**이 경사하강법으로 어떻게 학습되는지,
그리고 **다항 회귀·모델 규제·조기 종료**로 과소적합과 과대적합의 균형을 어떻게 맞추는지 살펴보았다.
다음 프로젝트(P2)에서는 회귀가 아닌 **분류** 문제로 넘어간다.